# CodonTransformer csi_top10_hc formal fine-tuning on Google Colab

This notebook runs only the primary `csi_top10_hc` experiment. It trains on 4,524 records, validates on 531 records, writes best/last checkpoints and all metadata to Google Drive, supports resume from its own `last.ckpt`, and compares the best checkpoint with the immutable official baseline on the independent 594-record test split. It does not train `all_clean_hc` or `csi_top25_hc`.

## 1. Require an NVIDIA T4-class CUDA runtime
Select **Runtime → Change runtime type → T4 GPU** before continuing.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU is required"
GPU_NAME = torch.cuda.get_device_name(0)
assert "T4" in GPU_NAME, f"Expected a T4 GPU, found: {GPU_NAME}"
print("CUDA device:", GPU_NAME)

## 2. Configure immutable sources and separate Drive paths
The repository is private. Store a fine-grained read-only GitHub token as the Colab secret `GITHUB_TOKEN`; never paste the token into a cell.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("GITHUB_REPO_URL", "https://github.com/Yuano0o/codontransformer-nb.git")
COLAB_WORKSPACE = Path(os.environ.get("COLAB_WORKSPACE", "/content"))
PROJECT_DIR = COLAB_WORKSPACE / "codontransformer-project"
UPSTREAM_REPO_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
HF_MODEL_ID = "adibvafa/CodonTransformer"
HF_MODEL_REVISION = "9744dcc920d813066391fc828d7a590207f148e8"
DRIVE_MOUNT = Path(os.environ.get("COLAB_DRIVE_MOUNT", "/content/drive"))
DRIVE_ROOT_RELATIVE = Path("MyDrive") / "CodonTransformer"
DATA_BASE_RELATIVE = Path("data") / "stage2" / "final_csi_cohorts" / "experiments" / "csi_top10_hc"
RUN_NAME = os.environ.get("COLAB_RUN_NAME", "finetune_csi_top10_hc_formal_v1")
AUTO_RESUME = True
EXPECTED_COUNTS = {"train": 4524, "validation": 531, "test": 594}
assert REPO_URL.startswith("https://github.com/")

## 3. Clone or fast-forward the private project and pin upstream source

In [ ]:
import subprocess
import tempfile
from google.colab import userdata

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    raise RuntimeError("Colab secret GITHUB_TOKEN is unavailable") from None
if not github_token:
    raise RuntimeError("Colab secret GITHUB_TOKEN is empty")
with tempfile.TemporaryDirectory(prefix="git-askpass-") as askpass_dir:
    askpass_path = Path(askpass_dir) / "askpass.sh"
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *sername*) printf \'%s\\n\' \'x-access-token\' ;;\n'
        '  *assword*) printf \'%s\\n\' "$COLAB_GITHUB_TOKEN" ;;\n'
        '  *) exit 1 ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)
    clone_env = os.environ.copy()
    clone_env.update({
        "GIT_ASKPASS": str(askpass_path),
        "GIT_TERMINAL_PROMPT": "0",
        "COLAB_GITHUB_TOKEN": github_token,
    })
    try:
        if not (PROJECT_DIR / ".git").is_dir():
            subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True, env=clone_env)
        else:
            subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True, env=clone_env)
    finally:
        clone_env.pop("COLAB_GITHUB_TOKEN", None)
        github_token = None
os.chdir(PROJECT_DIR)
upstream_dir = PROJECT_DIR / "upstream"
if not (upstream_dir / ".git").is_dir():
    subprocess.run(["git", "clone", UPSTREAM_REPO_URL, str(upstream_dir)], check=True)
subprocess.run(["git", "-C", str(upstream_dir), "checkout", UPSTREAM_COMMIT], check=True)
print("Project:", PROJECT_DIR)
print("Upstream commit:", UPSTREAM_COMMIT)

## 4. Install dependencies without replacing Colab CUDA PyTorch

In [ ]:
subprocess.run([os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
UPSTREAM_DIR = (PROJECT_DIR / "upstream").resolve()
assert (UPSTREAM_DIR / "CodonTransformer").is_dir()
upstream_pythonpath = str(UPSTREAM_DIR)
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = os.pathsep.join(value for value in (upstream_pythonpath, existing_pythonpath) if value)
if upstream_pythonpath not in os.sys.path:
    os.sys.path.insert(0, upstream_pythonpath)
import CodonTransformer
assert Path(CodonTransformer.__file__).resolve().is_relative_to(UPSTREAM_DIR)
print("Using pinned upstream source:", CodonTransformer.__file__)

## 5. Mount Drive and require all three leak-resistant splits
Upload the full `train.jsonl`, `validation.jsonl`, and `test.jsonl` before running this cell. The test split is not passed to training.

In [ ]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT))
DRIVE_ROOT = DRIVE_MOUNT / DRIVE_ROOT_RELATIVE
DATA_BASE = DRIVE_ROOT / DATA_BASE_RELATIVE
TRAIN_PATH = DATA_BASE / "train.jsonl"
VALIDATION_PATH = DATA_BASE / "validation.jsonl"
TEST_PATH = DATA_BASE / "test.jsonl"
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
def nonempty_lines(path):
    assert path.is_file(), f"Missing required dataset: {path}"
    with path.open(encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())
for split, path in (("train", TRAIN_PATH), ("validation", VALIDATION_PATH), ("test", TEST_PATH)):
    count = nonempty_lines(path)
    assert count == EXPECTED_COUNTS[split], f"{split}: expected {EXPECTED_COUNTS[split]}, found {count}"
    print(split, count, path)
print("Persistent formal run directory:", RUN_DIR)

## 6. Download the exact verified official pretrained snapshot
The immutable baseline is downloaded only to temporary Colab storage. Formal outputs use a separate Drive directory.

In [ ]:
MODEL_DIR = PROJECT_DIR / "models" / "pretrained"
subprocess.run([
    os.sys.executable, "scripts/download_pretrained.py",
    "--repo-id", HF_MODEL_ID,
    "--revision", HF_MODEL_REVISION,
    "--output-dir", str(MODEL_DIR),
], check=True)
assert (MODEL_DIR / "model.safetensors").is_file()
print("Verified official revision:", HF_MODEL_REVISION)

## 7. Detect a resumable checkpoint in the formal run only
Set `AUTO_RESUME = False` in the configuration cell only when intentionally starting a new run directory. The smoke-test directory is never inspected.

In [ ]:
LAST_CHECKPOINT = RUN_DIR / "checkpoints" / "last.ckpt"
RESUME_CHECKPOINT = LAST_CHECKPOINT if AUTO_RESUME and LAST_CHECKPOINT.is_file() else None
print("Resume checkpoint:", RESUME_CHECKPOINT or "none; initialize from official pretrained weights")

## 8. Run formal csi_top10_hc training
This executes at most five epochs with batch size 1, gradient accumulation 8, deterministic validation, best/last checkpointing, gradient clipping, early stopping, and resume support.

In [ ]:
training_command = [
    os.sys.executable, "scripts/finetune_codontransformer.py",
    "--config", "configs/finetune_cuda_csi_top10.yaml",
    "--model-dir", str(MODEL_DIR),
    "--dataset-path", str(TRAIN_PATH),
    "--validation-dataset-path", str(VALIDATION_PATH),
    "--test-dataset-path", str(TEST_PATH),
    "--output-dir", str(RUN_DIR),
]
if RESUME_CHECKPOINT is not None:
    training_command.extend(["--resume-from-checkpoint", str(RESUME_CHECKPOINT)])
subprocess.run(training_command, check=True)
import json
TRAINING_RESULT_PATH = RUN_DIR / "training_result.json"
training_result = json.loads(TRAINING_RESULT_PATH.read_text(encoding="utf-8"))
BEST_CHECKPOINT = Path(training_result["best_checkpoint"])
LAST_CHECKPOINT = Path(training_result["last_checkpoint"])
assert BEST_CHECKPOINT.is_file()
assert LAST_CHECKPOINT.is_file()
training_result

## 9. Compare official baseline and best fine-tuned checkpoint on all 594 test records
Both models receive identical deterministic masks. The report contains masked-token NLL, perplexity, top-1 accuracy, top-3 accuracy, and fine-tuned-minus-baseline deltas.

In [ ]:
TEST_REPORT_PATH = RUN_DIR / "test_baseline_vs_finetuned.json"
subprocess.run([
    os.sys.executable, "scripts/evaluate_codontransformer_test.py",
    "--model-dir", str(MODEL_DIR),
    "--checkpoint", str(BEST_CHECKPOINT),
    "--test-dataset", str(TEST_PATH),
    "--output", str(TEST_REPORT_PATH),
    "--device", "cuda",
    "--batch-size", "1",
    "--mask-probability", "0.15",
    "--mask-seed", "2025",
    "--expected-records", "594",
], check=True)
test_report = json.loads(TEST_REPORT_PATH.read_text(encoding="utf-8"))
assert test_report["baseline"]["records"] == 594
assert test_report["finetuned"]["records"] == 594
test_report

## 10. Reload best checkpoint and verify one DNA generation

In [ ]:
INFERENCE_REPORT_PATH = RUN_DIR / "best_checkpoint_inference_validation.json"
subprocess.run([
    os.sys.executable, "scripts/validate_checkpoint_inference.py",
    "--model-dir", str(MODEL_DIR),
    "--checkpoint", str(BEST_CHECKPOINT),
    "--output", str(INFERENCE_REPORT_PATH),
    "--device", "cuda",
    "--organism", "Nicotiana tabacum",
], check=True)
inference_report = json.loads(INFERENCE_REPORT_PATH.read_text(encoding="utf-8"))
assert inference_report["translation_verified"] is True
inference_report

## 11. Inspect persistent Drive outputs
Before disconnecting, confirm that best and last checkpoints, Lightning logs, `training.log`, `resolved_config.yaml`, `runtime.json`, `training_result.json`, and both evaluation reports are present.

In [ ]:
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN_DIR), path.stat().st_size)